# Dataset & DataLoader

In [1]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T

from dataset import TrainDataset, TestDataset

In [2]:
# hyperparams
image_size = 64
batch_size_train = 32 # Iteration 극대화
batch_size_eval = 128 # 평가 시간 단축
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)
val_ratio = 0.2 # train 80%, val 20%

In [3]:
# data augmentation
train_transform = T.Compose([
    T.RandomResizedCrop(image_size, scale=(0.9, 1.0), ratio=(0.9, 1.1)), # Crop 비율을 90% 이상으로 유지
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_dataset = TrainDataset(root_path = './cs441-assn3-data/Train_64/', transform = train_transform)
test_dataset = TestDataset(root_path = './cs441-assn3-data/Test_64/', transform = eval_transform)

In [4]:
# train/val splits
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_dataset_split, val_dataset = random_split(
    train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(dataset=train_dataset_split,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=4,
    drop_last = True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_eval,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(dataset=test_dataset,
    batch_size=batch_size_eval,
    shuffle=False,
    num_workers=4
)

# Your Awesome Model

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from convnext import ConvNeXt
from resnext import ResNeXt34

from utils import *

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.__version__)
print(device)

2.5.1+cu121
cuda


In [ ]:
class StackingEnsembleModel(nn.Module):
    """
    ResNeXt34와 ConvNeXt의 출력을 Meta-model (Stacking Layer)을 통해 학습하여 앙상블합니다.
    """
    def __init__(self, num_classes=15):
        super().__init__()
        self.num_classes = num_classes
        
        self.resnext = ResNeXt34(num_classes=num_classes, cardinality=16, base_width=64)
        self.convnext = ConvNeXt(num_classes=num_classes, drop_path_rate=0.05)
        
        self.stacking_layer = nn.Linear(num_classes * 2, num_classes)
        
        nn.init.normal_(self.stacking_layer.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.stacking_layer.bias, 0)
        
    def forward(self, x, x2=None):
        if self.training:
            out1 = self.resnext(x)
            if x2 is not None:
                out2 = self.convnext(x2)
            else:
                out2 = self.convnext(x)
            
            stacked_output = torch.cat((out1, out2), dim=1)
            final_output = self.stacking_layer(stacked_output)
            return final_output

        with torch.no_grad():
            out1_orig = self.resnext(x)
            out2_orig = self.convnext(x)
            
            stacked_orig = torch.cat((out1_orig, out2_orig), dim=1)
            output_orig = self.stacking_layer(stacked_orig)
            
            x_flipped = torch.flip(x, dims=[-1]) 
            
            out1_flip = self.resnext(x_flipped)
            out2_flip = self.convnext(x_flipped)
            
            stacked_flip = torch.cat((out1_flip, out2_flip), dim=1)
            output_flip = self.stacking_layer(stacked_flip)
            
            final_output = (output_orig + output_flip) / 2
            
            return final_output

        ''' 좌우반전 + 4방향 TTA
        with torch.no_grad():    
            all_outputs = []
            
            for k in range(4):
                x_rotated = torch.rot90(x, k, dims=[-2, -1])
                
                out1 = self.resnext(x_rotated)
                out2 = self.convnext(x_rotated)
                stacked = torch.cat((out1, out2), dim=1)
                all_outputs.append(self.stacking_layer(stacked))
                
                x_flipped = torch.flip(x_rotated, dims=[-1])
                out1_flip = self.resnext(x_flipped)
                out2_flip = self.convnext(x_flipped)
                stacked_flip = torch.cat((out1_flip, out2_flip), dim=1)
                all_outputs.append(self.stacking_layer(stacked_flip))
                
            final_output = torch.stack(all_outputs, dim=0).mean(dim=0)
            
            return final_output
        '''

In [8]:
model = StackingEnsembleModel(num_classes=15).to(device)

# Model parameter checking

In [9]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 71023835
Parameter usage : 71.023835%


# Model training

In [10]:
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import SequentialLR, LambdaLR, CosineAnnealingLR

In [11]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

GPU count: 1
Current device index: 0
Current device name: NVIDIA GeForce RTX 4080 SUPER


In [12]:
import tqdm
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR
from torch.cuda.amp import autocast

# 설정
epochs = 15
save_path = "best_ensemble_model.pth"
best_val_loss = float("inf")

# Optimizer & Scheduler
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.05)
scaler = torch.amp.GradScaler('cuda')

# Scheduler: Warmup (3.5 epochs) + Cosine
total_steps = epochs * len(train_loader)
warmup_steps = 500
scheduler = SequentialLR(
    optimizer, 
    schedulers=[
        LambdaLR(optimizer, lambda step: (step + 1) / warmup_steps),
        CosineAnnealingLR(optimizer, T_max=total_steps + 500)
    ], 
    milestones=[warmup_steps]
)

print("Start Training Ensemble Model...")

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss, train_acc = 0.0, 0.0
    total = 0
    
    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            output = model(x)
            loss = criterion(output, y)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step() # Step per batch
        
        train_loss += loss.item() * y.size(0)
        train_acc += (output.argmax(1) == y).sum().item()
        total += y.size(0)
        
    print(f"Epoch {epoch} | Train Loss: {train_loss/total:.4f} | Acc: {train_acc/total:.4f}")

    # VALIDATION
    model.eval()
    val_loss, val_acc = 0.0, 0.0
    total = 0
    
    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            x, y = x.to(device), y.to(device)
            with autocast():
                output = model(x)
                loss = criterion(output, y)
            val_loss += loss.item() * y.size(0)
            val_acc += (output.argmax(1) == y).sum().item()
            total += y.size(0)
            
    val_loss /= total
    val_acc /= total
    print(f"Epoch {epoch} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"Saved Best Model (Val Loss: {val_loss:.4f})")

Start Training Ensemble Model...


Epoch 0 [Train]:  44%|████▍     | 499/1125 [02:45<03:14,  3.22it/s]c:\Users\user\miniconda3\envs\jupyter\lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)
Epoch 0 [Train]: 100%|██████████| 1125/1125 [06:02<00:00,  3.10it/s]


Epoch 0 | Train Loss: 2.2651 | Acc: 0.2590


Epoch 0 [Val]:   0%|          | 0/71 [00:00<?, ?it/s]C:\Users\user\AppData\Local\Temp\ipykernel_1944\501664833.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 0 [Val]: 100%|██████████| 71/71 [02:59<00:00,  2.53s/it]


Epoch 0 | Val Loss: 1.9860 | Acc: 0.3638
Saved Best Model (Val Loss: 1.9860)


...

Epoch 13 | Val Loss: 1.0364 | Acc: 0.6608
Saved Best Model (Val Loss: 1.0364)


Epoch 14 [Train]: 100%|██████████| 1125/1125 [06:01<00:00,  3.12it/s]


Epoch 14 | Train Loss: 0.8167 | Acc: 0.7335


Epoch 14 [Val]: 100%|██████████| 71/71 [02:59<00:00,  2.53s/it]

Epoch 14 | Val Loss: 1.0371 | Acc: 0.6539


In [13]:
'''
checkpoint = torch.load("best_ensemble_model.pth", map_location=device)

# 먼저 순수 모델을 만들고 로드
base_model = StackingEnsembleModel(num_classes=15)
base_model.load_state_dict(checkpoint)

# 그 다음에 DataParallel로 감쌈
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(device)
'''

'\ncheckpoint = torch.load("best_ensemble_model.pth", map_location=device)\n\n# 먼저 순수 모델을 만들고 로드\nbase_model = StackingEnsembleModel(num_classes=15)\nbase_model.load_state_dict(checkpoint)\n\n# 그 다음에 DataParallel로 감쌈\nif torch.cuda.device_count() > 1:\n    model = torch.nn.DataParallel(base_model)\nelse:\n    model = base_model\n\nmodel = model.to(device)\n'

# Submit
Do not edit the submission code below.

In [14]:
import pandas as pd

# Load Best Model
submit = pd.read_csv('./cs441-assn3-data/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 71023835
Parameter usage : 71.023835%


100%|██████████| 59/59 [12:50<00:00, 13.06s/it]
